# AlphaZero — Chess Supervised Pre-training on Colab

Phase 2 of Sub-project 3b: bootstrap a 15M-param `AlphaZeroNet` by behavior-cloning Stockfish moves on a corpus of Stockfish-vs-Stockfish games.

**Expected wall-clock on a free-tier T4 GPU:** ~25–30 minutes for ~3M positions × 3 epochs.

**Inputs:** a tar.gz of `data/chess_corpus/` (typically ~60 MB), uploaded to your Google Drive and shared with anyone-with-link viewer permission.

**Outputs:** `pretrained.pt` (~60 MB), saved back to your Drive at the end.

**Prerequisites — see [`notebooks/README.md`](https://github.com/venkata91/alphazero/blob/main/notebooks/README.md) for the Azure VM → Drive transfer steps before running this notebook.**

## Plan
1. Verify GPU runtime
2. Clone the repo + install deps
3. Download the corpus tarball from Drive (via `gdown`)
4. Sanity-check the corpus
5. Run pretrain (~25 min on T4)
6. Save `pretrained.pt` back to Drive

## Step 1 — Verify GPU runtime

**Before running:** Runtime → Change runtime type → **T4 GPU** (free tier).

If the cell below prints `CPU only`, stop and switch the runtime.

In [ ]:
!nvidia-smi | head -10
import torch
print()
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:        ', torch.cuda.get_device_name(0))
    print('Memory:        ', f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('Device:         CPU only — STOP and switch runtime to GPU')

## Step 2 — Clone the repo

Pulls the `alphazero` codebase. If you reconnect later, the clone is skipped.

In [ ]:
import os

if not os.path.isdir('/content/alphazero'):
    !git clone --depth 1 https://github.com/venkata91/alphazero.git /content/alphazero
else:
    print('Repo already cloned — skipping.')

%cd /content/alphazero
!git log --oneline -3

## Step 3 — Install dependencies

`pip install -e .` puts the `alphazero` package on the Python path. The repo already lists `torch`, `numpy`, `python-chess`, `tomli`, etc. as deps. Colab's preinstalled torch satisfies the version constraint, so this is mostly a no-op.

In [ ]:
!pip install -q -e ".[dev]"
import alphazero
print('alphazero package imported OK')
print('numpy, chess, torch imports:')
import numpy as np
import chess
import torch
print(f'  torch={torch.__version__}, cuda={torch.version.cuda}')
print(f'  numpy={np.__version__}')
print(f'  chess={chess.__version__}')

## Step 4 — Download the corpus from Drive

**Setup required before running:**

1. On your Azure VM, tar the corpus:
   ```bash
   cd ~/git/alphazero
   tar czf chess_corpus.tar.gz data/chess_corpus/
   ls -lh chess_corpus.tar.gz
   ```
2. Copy to your laptop:
   ```bash
   scp <user>@<azure-vm>:~/git/alphazero/chess_corpus.tar.gz ./
   ```
3. Upload `chess_corpus.tar.gz` to Google Drive (drag-drop to drive.google.com).
4. Right-click the file in Drive → **Share** → **Anyone with the link** (viewer).
5. Copy the share link — it looks like `https://drive.google.com/file/d/<FILE_ID>/view?usp=sharing`.
6. **Paste just the `<FILE_ID>` (between `/d/` and `/view`) into the cell below.**

Example: if your link is `https://drive.google.com/file/d/1aBcDeFgHiJkLmNoPqRsTuVwXyZ/view?usp=sharing`, the file ID is `1aBcDeFgHiJkLmNoPqRsTuVwXyZ`.

In [ ]:
# === PASTE YOUR DRIVE FILE ID BELOW ===
CORPUS_FILE_ID = 'PASTE_FILE_ID_HERE'
# ======================================

assert CORPUS_FILE_ID != 'PASTE_FILE_ID_HERE', 'Edit the cell to paste your Drive file ID first.'

!pip install -q gdown
!mkdir -p data
!cd data && gdown --id {CORPUS_FILE_ID} -O chess_corpus.tar.gz

print()
print('Extracting tarball...')
!cd data && tar xzf chess_corpus.tar.gz && rm chess_corpus.tar.gz
!ls data/chess_corpus | head -5
!echo '...'
!ls data/chess_corpus | wc -l
!du -sh data/chess_corpus/

## Step 5 — Sanity-check the corpus

Verifies the shard count, position count, and win/draw/loss distribution. Catches truncated uploads or corrupted files.

In [ ]:
import numpy as np
from pathlib import Path

shards = sorted(Path('data/chess_corpus').glob('shard_*.npz'))
print(f'shards: {len(shards)}')

total = 0
outcomes = []
for s in shards:
    data = np.load(s)
    total += data['states'].shape[0]
    outcomes.append(data['outcomes'])
outcomes = np.concatenate(outcomes)

print(f'positions: {total:,}')
print(f'wins:   {int((outcomes==1).sum()):>8,}  ({(outcomes==1).mean()*100:5.1f}%)')
print(f'draws:  {int((outcomes==0).sum()):>8,}  ({(outcomes==0).mean()*100:5.1f}%)')
print(f'losses: {int((outcomes==-1).sum()):>8,}  ({(outcomes==-1).mean()*100:5.1f}%)')

# Expected: ~15-20% wins, ~60-70% draws, ~15-20% losses (wins~losses tied to <2%)
decisive = (outcomes != 0).sum() / len(outcomes)
print(f'\ndecisive: {decisive*100:.1f}% — expect 30-40%')
assert abs((outcomes==1).mean() - (outcomes==-1).mean()) < 0.05, 'wins and losses should be roughly balanced'
print('\n✓ Corpus looks healthy.')

## Step 6 — Run pre-training

Trains a 15M-param `AlphaZeroNet` on the corpus for up to 3 epochs with early stopping (patience=2 on validation loss). Saves the best-by-val-loss checkpoint as `pretrained.pt`.

**Wall-clock: ~25-30 minutes on T4** (the GPU should be saturated; check the cell output for per-epoch elapsed time).

Expected output per epoch:
```
epoch 1/3: train_loss=4.8 val_loss=4.5 lr=5.0e-04 elapsed=~500s
epoch 2/3: train_loss=3.1 val_loss=3.0 lr=2.5e-04 elapsed=~500s
epoch 3/3: train_loss=2.6 val_loss=2.7 lr=5.0e-05 elapsed=~500s
Done. best_val_loss=2.7
```

If val_loss diverges (gets worse) for 2 consecutive epochs, training stops early.

In [ ]:
!python3 -m alphazero pretrain --config configs/chess-pretrain.toml

## Step 7 — Save `pretrained.pt` back to Drive

Mounts your Drive and copies the checkpoint to a `colab_output/` folder. Reconnect after a session disconnect by re-running just this notebook — the file is safe in Drive.

Browser will pop up an OAuth prompt the first time. Authorize Drive read+write.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
out_dir = '/content/drive/MyDrive/alphazero/colab_output'
os.makedirs(out_dir, exist_ok=True)

!cp pretrained.pt {out_dir}/pretrained.pt
!ls -lh {out_dir}/pretrained.pt

## Done!

`pretrained.pt` is in your Drive at `MyDrive/alphazero/colab_output/pretrained.pt`.

## Next: Phase 3 AZ refinement

On your Azure VM:

```bash
# 1. Download pretrained.pt back from Drive (manually via drive.google.com, then scp)
scp <laptop>:~/Downloads/pretrained.pt <user>@<azure-vm>:~/git/alphazero/

# 2. Verify the file landed
ls -lh ~/git/alphazero/pretrained.pt

# 3. Run refinement (in tmux — 6-10 hour run)
cd ~/git/alphazero
tmux new -s refine
python3 -m alphazero train --game chess --config configs/chess-refine.toml --resume-from pretrained.pt
```

After 10 iterations finishes:

```bash
python3 -m alphazero eval --game chess --checkpoint checkpoints_refine/iter_0010.pt --num-games 100
```

Pass criterion: `win_rate >= 0.50` vs Stockfish ELO=1500.